# UdaPlay - Part 1: RAG Pipeline

In this notebook, we build the Retrieval Augmented Generation (RAG) pipeline for the UdaPlay gaming assistant. This includes:

1. Loading and processing game data from JSON files
2. Setting up a ChromaDB vector database
3. Embedding game data into the vector store
4. Implementing semantic search functionality
5. Creating a reusable vector store manager

## 1. Environment Setup

In [1]:
# Load environment variables
from dotenv import load_dotenv
import os

load_dotenv("config.env")

assert os.getenv("OPENAI_API_KEY") is not None, "OPENAI_API_KEY not found in config.env"
assert os.getenv("TAVILY_API_KEY") is not None, "TAVILY_API_KEY not found in config.env"

OPENAI_BASE_URL = os.getenv(
    "OPENAI_BASE_URL",
    "https://openai.vocareum.com/v1"
)

print("Environment variables loaded successfully!")
print(f"OpenAI Base URL: {OPENAI_BASE_URL}")

Environment variables loaded successfully!
OpenAI Base URL: https://openai.vocareum.com/v1


In [2]:
# Import required libraries
import json
import chromadb
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Dict, Optional
import hashlib

## 2. Load and Process Game Data

We load game information from JSON files that contain details about video games and game companies.

In [3]:
# Define data models using Pydantic
class Game(BaseModel):
    """Model representing a video game."""
    title: str
    developer: str
    publisher: str
    release_date: str
    platforms: List[str]
    genre: str
    description: str

class Company(BaseModel):
    """Model representing a game company."""
    name: str
    headquarters: str
    founded: str
    notable_franchises: List[str]
    current_projects: str
    description: str

In [4]:
def load_games(filepath: str) -> List[Game]:
    """Load games from a JSON file."""
    with open(filepath, 'r') as f:
        data = json.load(f)
    return [Game(**game) for game in data]

def load_companies(filepath: str) -> List[Company]:
    """Load companies from a JSON file."""
    with open(filepath, 'r') as f:
        data = json.load(f)
    return [Company(**company) for company in data]

# Load the data
games = load_games("data/games/games.json")
companies = load_companies("data/games/companies.json")

print(f"Loaded {len(games)} games and {len(companies)} companies")
print("\nGames:")
for game in games:
    print(f"  - {game.title} ({game.developer})")
print("\nCompanies:")
for company in companies:
    print(f"  - {company.name}")

Loaded 15 games and 10 companies

Games:
  - FIFA 21 (EA Vancouver)
  - God of War Ragnarok (Santa Monica Studio)
  - Pokemon Red (Game Freak)
  - Grand Theft Auto V (Rockstar North)
  - The Legend of Zelda: Breath of the Wild (Nintendo EPD)
  - The Witcher 3: Wild Hunt (CD Projekt Red)
  - Minecraft (Mojang Studios)
  - Elden Ring (FromSoftware)
  - Red Dead Redemption 2 (Rockstar Studios)
  - Cyberpunk 2077 (CD Projekt Red)
  - Dispatch (AdHoc Studio)
  - Stray (BlueTwelve Studio)
  - Esoteric Ebb (Christoffer Bodegard)
  - Battlefield 6 (Battlefield Studios (DICE, Criterion Games, Motive Studio, Ripple Effect Studios))
  - Forza Motorsport (Turn 10 Studios)

Companies:
  - Electronic Arts
  - Rockstar Games
  - Nintendo
  - Sony Interactive Entertainment
  - CD Projekt
  - FromSoftware
  - Mojang Studios
  - Bandai Namco Entertainment
  - Annapurna Interactive
  - Raw Fury


## 3. Process Data for Vector Database

We convert each game and company into a rich text document that captures all relevant information for semantic search.

In [5]:
def game_to_document(game: Game) -> str:
    """Convert a Game object to a searchable text document."""
    platforms_str = ", ".join(game.platforms)
    return (
        f"Game Title: {game.title}\n"
        f"Developer: {game.developer}\n"
        f"Publisher: {game.publisher}\n"
        f"Release Date: {game.release_date}\n"
        f"Platforms: {platforms_str}\n"
        f"Genre: {game.genre}\n"
        f"Description: {game.description}"
    )

def company_to_document(company: Company) -> str:
    """Convert a Company object to a searchable text document."""
    franchises_str = ", ".join(company.notable_franchises)
    return (
        f"Company Name: {company.name}\n"
        f"Headquarters: {company.headquarters}\n"
        f"Founded: {company.founded}\n"
        f"Notable Franchises: {franchises_str}\n"
        f"Current Projects: {company.current_projects}\n"
        f"Description: {company.description}"
    )

def generate_id(text: str) -> str:
    """Generate a unique ID for a document based on its content."""
    return hashlib.md5(text.encode()).hexdigest()

# Process all documents
documents = []
metadatas = []
ids = []

# Process games
for game in games:
    doc = game_to_document(game)
    documents.append(doc)
    metadatas.append({
        "type": "game",
        "title": game.title,
        "developer": game.developer,
        "publisher": game.publisher,
        "release_date": game.release_date,
        "genre": game.genre
    })
    ids.append(f"game_{generate_id(game.title)}")

# Process companies
for company in companies:
    doc = company_to_document(company)
    documents.append(doc)
    metadatas.append({
        "type": "company",
        "name": company.name,
        "headquarters": company.headquarters,
        "founded": company.founded
    })
    ids.append(f"company_{generate_id(company.name)}")

print(f"Prepared {len(documents)} documents for embedding")
print(f"\nSample game document:\n{'-'*50}")
print(documents[0])
print(f"\n{'='*50}")
print(f"\nSample company document:\n{'-'*50}")
print(documents[len(games)])

Prepared 25 documents for embedding

Sample game document:
--------------------------------------------------
Game Title: FIFA 21
Developer: EA Vancouver
Publisher: Electronic Arts
Release Date: October 9, 2020
Platforms: PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S, PC, Nintendo Switch, Stadia
Genre: Sports, Football Simulation
Description: FIFA 21 is a football simulation video game published by Electronic Arts. It is the 28th installment in the FIFA series. The game features improved gameplay mechanics, enhanced AI, and new game modes including an updated Career Mode and the return of FIFA Ultimate Team.


Sample company document:
--------------------------------------------------
Company Name: Electronic Arts
Headquarters: Redwood City, California, USA
Founded: 1982
Notable Franchises: FIFA, Madden NFL, Battlefield, The Sims, Need for Speed, Apex Legends
Current Projects: EA is currently working on EA Sports FC series (successor to FIFA), the next Battlefield game, and e

## 4. Set Up ChromaDB Vector Database

We create a persistent ChromaDB instance that will store our game data with embeddings for semantic search.

In [6]:
class VectorStoreManager:
    """Manages the ChromaDB vector store for game information."""
    
    def __init__(self, persist_directory: str = "./chroma_db", collection_name: str = "udaplay_games"):
        """
        Initialize the vector store manager.
        
        Args:
            persist_directory: Directory to persist the ChromaDB data
            collection_name: Name of the collection in ChromaDB
        """
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        
        # Initialize OpenAI client for embeddings
        self.openai_client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=OPENAI_BASE_URL
        )
        
        # Initialize ChromaDB with persistent storage
        self.chroma_client = chromadb.PersistentClient(path=persist_directory)
        
        # Get or create the collection
        self.collection = self.chroma_client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        
        print(f"Vector store initialized. Collection '{collection_name}' has {self.collection.count()} documents.")
    
    def get_embeddings(self, texts: List[str]) -> List[List[float]]:
        """
        Generate embeddings for a list of texts using OpenAI's API.
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            List of embedding vectors
        """
        response = self.openai_client.embeddings.create(
            model="text-embedding-3-small",
            input=texts
        )
        return [item.embedding for item in response.data]
    
    def add_documents(self, documents: List[str], metadatas: List[Dict], ids: List[str]):
        """
        Add documents to the vector store with embeddings.
        
        Args:
            documents: List of document texts
            metadatas: List of metadata dictionaries
            ids: List of unique document IDs
        """
        # Generate embeddings
        print(f"Generating embeddings for {len(documents)} documents...")
        embeddings = self.get_embeddings(documents)
        
        # Add to ChromaDB
        self.collection.upsert(
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas,
            ids=ids
        )
        
        print(f"Successfully added {len(documents)} documents to the vector store.")
        print(f"Total documents in collection: {self.collection.count()}")
    
    def search(self, query: str, n_results: int = 3, filter_type: Optional[str] = None) -> Dict:
        """
        Perform semantic search on the vector store.
        
        Args:
            query: The search query text
            n_results: Number of results to return
            filter_type: Optional filter for document type ('game' or 'company')
            
        Returns:
            Dictionary containing search results with documents, metadatas, and distances
        """
        # Generate query embedding
        query_embedding = self.get_embeddings([query])[0]
        
        # Build query parameters
        query_params = {
            "query_embeddings": [query_embedding],
            "n_results": n_results,
            "include": ["documents", "metadatas", "distances"]
        }
        
        if filter_type:
            query_params["where"] = {"type": filter_type}
        
        # Perform the search
        results = self.collection.query(**query_params)
        
        return {
            "documents": results["documents"][0] if results["documents"] else [],
            "metadatas": results["metadatas"][0] if results["metadatas"] else [],
            "distances": results["distances"][0] if results["distances"] else []
        }
    
    def get_collection_info(self) -> Dict:
        """Get information about the current collection."""
        return {
            "name": self.collection_name,
            "count": self.collection.count(),
            "persist_directory": self.persist_directory
        }

print("VectorStoreManager ready.")

VectorStoreManager ready.


## 5. Initialize Vector Store and Add Data

In [7]:
# Initialize the vector store manager
vector_store = VectorStoreManager(
    persist_directory="./chroma_db",
    collection_name="udaplay_games"
)

# Add all documents to the vector store
vector_store.add_documents(documents, metadatas, ids)

# Display collection info
info = vector_store.get_collection_info()
print(f"\nCollection Info:")
print(f"  Name: {info['name']}")
print(f"  Document Count: {info['count']}")
print(f"  Persist Directory: {info['persist_directory']}")

Vector store initialized. Collection 'udaplay_games' has 0 documents.
Generating embeddings for 25 documents...


Successfully added 25 documents to the vector store.
Total documents in collection: 25

Collection Info:
  Name: udaplay_games
  Document Count: 25
  Persist Directory: ./chroma_db


## 6. Demonstrate Semantic Search

Let's test the semantic search functionality with various queries to demonstrate the RAG pipeline works correctly.

In [8]:
def display_search_results(query: str, results: Dict):
    """Display search results in a formatted way."""
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    for i, (doc, metadata, distance) in enumerate(zip(
        results['documents'], results['metadatas'], results['distances']
    )):
        similarity = 1 - distance  # Convert distance to similarity score
        print(f"\n--- Result {i+1} (Similarity: {similarity:.4f}) ---")
        print(f"Type: {metadata.get('type', 'unknown')}")
        if metadata.get('type') == 'game':
            print(f"Title: {metadata.get('title', 'N/A')}")
        else:
            print(f"Name: {metadata.get('name', 'N/A')}")
        print(f"Document Preview: {doc[:200]}...")
    
    print(f"\n{'='*60}")

In [9]:
# Test Query 1: Who developed FIFA 21?
query1 = "Who developed FIFA 21?"
results1 = vector_store.search(query1, n_results=3)
display_search_results(query1, results1)


Query: Who developed FIFA 21?

--- Result 1 (Similarity: 0.7340) ---
Type: game
Title: FIFA 21
Document Preview: Game Title: FIFA 21
Developer: EA Vancouver
Publisher: Electronic Arts
Release Date: October 9, 2020
Platforms: PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S, PC, Nintendo Switch, Stadia
Gen...

--- Result 2 (Similarity: 0.4719) ---
Type: company
Name: Electronic Arts
Document Preview: Company Name: Electronic Arts
Headquarters: Redwood City, California, USA
Founded: 1982
Notable Franchises: FIFA, Madden NFL, Battlefield, The Sims, Need for Speed, Apex Legends
Current Projects: EA i...

--- Result 3 (Similarity: 0.4324) ---
Type: game
Title: Forza Motorsport
Document Preview: Game Title: Forza Motorsport
Developer: Turn 10 Studios
Publisher: Xbox Game Studios
Release Date: October 10, 2023
Platforms: PC, Xbox Series X/S
Genre: Sim Racing
Description: Forza Motorsport (2023...



In [10]:
# Test Query 2: When was God of War Ragnarok released?
query2 = "When was God of War Ragnarok released?"
results2 = vector_store.search(query2, n_results=3)
display_search_results(query2, results2)


Query: When was God of War Ragnarok released?

--- Result 1 (Similarity: 0.7162) ---
Type: game
Title: God of War Ragnarok
Document Preview: Game Title: God of War Ragnarok
Developer: Santa Monica Studio
Publisher: Sony Interactive Entertainment
Release Date: November 9, 2022
Platforms: PlayStation 4, PlayStation 5, PC
Genre: Action-Advent...

--- Result 2 (Similarity: 0.3567) ---
Type: game
Title: Elden Ring
Document Preview: Game Title: Elden Ring
Developer: FromSoftware
Publisher: Bandai Namco Entertainment
Release Date: February 25, 2022
Platforms: PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S, PC
Genre: Actio...

--- Result 3 (Similarity: 0.3488) ---
Type: game
Title: The Witcher 3: Wild Hunt
Document Preview: Game Title: The Witcher 3: Wild Hunt
Developer: CD Projekt Red
Publisher: CD Projekt
Release Date: May 19, 2015
Platforms: PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S, PC, Nintendo Switch
...



In [11]:
# Test Query 3: What platform was Pokemon Red launched on?
query3 = "What platform was Pokemon Red launched on?"
results3 = vector_store.search(query3, n_results=3)
display_search_results(query3, results3)


Query: What platform was Pokemon Red launched on?

--- Result 1 (Similarity: 0.7020) ---
Type: game
Title: Pokemon Red
Document Preview: Game Title: Pokemon Red
Developer: Game Freak
Publisher: Nintendo
Release Date: February 27, 1996
Platforms: Game Boy
Genre: Role-Playing Game (RPG)
Description: Pokemon Red is a role-playing video ga...

--- Result 2 (Similarity: 0.3507) ---
Type: company
Name: Nintendo
Document Preview: Company Name: Nintendo
Headquarters: Kyoto, Japan
Founded: 1889
Notable Franchises: Mario, The Legend of Zelda, Pokemon, Metroid, Animal Crossing, Super Smash Bros.
Current Projects: Nintendo is worki...

--- Result 3 (Similarity: 0.3074) ---
Type: game
Title: Red Dead Redemption 2
Document Preview: Game Title: Red Dead Redemption 2
Developer: Rockstar Studios
Publisher: Rockstar Games
Release Date: October 26, 2018
Platforms: PlayStation 4, Xbox One, PC, Stadia
Genre: Action-Adventure, Open Worl...



In [12]:
# Test Query 4: What is Rockstar Games working on right now?
query4 = "What is Rockstar Games working on right now?"
results4 = vector_store.search(query4, n_results=3)
display_search_results(query4, results4)


Query: What is Rockstar Games working on right now?

--- Result 1 (Similarity: 0.7037) ---
Type: company
Name: Rockstar Games
Document Preview: Company Name: Rockstar Games
Headquarters: New York City, New York, USA
Founded: 1998
Notable Franchises: Grand Theft Auto, Red Dead Redemption, Max Payne, Bully, L.A. Noire
Current Projects: Rockstar...

--- Result 2 (Similarity: 0.5577) ---
Type: game
Title: Grand Theft Auto V
Document Preview: Game Title: Grand Theft Auto V
Developer: Rockstar North
Publisher: Rockstar Games
Release Date: September 17, 2013
Platforms: PlayStation 3, PlayStation 4, PlayStation 5, Xbox 360, Xbox One, Xbox Ser...

--- Result 3 (Similarity: 0.5207) ---
Type: game
Title: Red Dead Redemption 2
Document Preview: Game Title: Red Dead Redemption 2
Developer: Rockstar Studios
Publisher: Rockstar Games
Release Date: October 26, 2018
Platforms: PlayStation 4, Xbox One, PC, Stadia
Genre: Action-Adventure, Open Worl...



In [13]:
# Test Query 5: Search only for companies
query5 = "Open world RPG games"
results5 = vector_store.search(query5, n_results=3, filter_type="game")
display_search_results(query5, results5)


Query: Open world RPG games

--- Result 1 (Similarity: 0.4687) ---
Type: game
Title: Cyberpunk 2077
Document Preview: Game Title: Cyberpunk 2077
Developer: CD Projekt Red
Publisher: CD Projekt
Release Date: December 10, 2020
Platforms: PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S, PC, Stadia
Genre: Action ...

--- Result 2 (Similarity: 0.4621) ---
Type: game
Title: The Witcher 3: Wild Hunt
Document Preview: Game Title: The Witcher 3: Wild Hunt
Developer: CD Projekt Red
Publisher: CD Projekt
Release Date: May 19, 2015
Platforms: PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S, PC, Nintendo Switch
...

--- Result 3 (Similarity: 0.4580) ---
Type: game
Title: Esoteric Ebb
Document Preview: Game Title: Esoteric Ebb
Developer: Christoffer Bodegard
Publisher: Raw Fury
Release Date: March 3, 2026
Platforms: PC
Genre: Role-Playing Game (RPG)
Description: Esoteric Ebb is a role-playing video ...



## 7. RAG Query Function

Combining the vector search with an LLM to generate natural language answers.

In [14]:
def rag_query(question: str, vector_store: VectorStoreManager, n_results: int = 3) -> str:
    """
    Perform a RAG query: retrieve relevant documents and generate an answer.
    
    Args:
        question: The user's question
        vector_store: The VectorStoreManager instance
        n_results: Number of documents to retrieve
        
    Returns:
        Generated answer string
    """
    # Retrieve relevant documents
    results = vector_store.search(question, n_results=n_results)
    
    # Build context from retrieved documents
    context = "\n\n---\n\n".join(results['documents'])
    
    # Generate answer using OpenAI
    client = OpenAI(
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=OPENAI_BASE_URL
    )
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are UdaPlay, a helpful gaming assistant. Answer the user's question "
                    "based on the provided context. If the context doesn't contain enough "
                    "information to fully answer the question, say so clearly. "
                    "Always cite your source (e.g., 'Based on internal game database...')."
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ],
        temperature=0.3
    )
    
    return response.choices[0].message.content

print("RAG query function ready.")

RAG query function ready.


In [15]:
# Test RAG query
print("Testing RAG Query Pipeline:")
print("=" * 60)

test_questions = [
    "Who developed FIFA 21?",
    "When was God of War Ragnarok released?",
    "What platform was Pokemon Red launched on?",
    "What is Rockstar Games working on right now?"
]

for question in test_questions:
    print(f"\nQ: {question}")
    answer = rag_query(question, vector_store)
    print(f"A: {answer}")
    print("-" * 60)

Testing RAG Query Pipeline:

Q: Who developed FIFA 21?


A: FIFA 21 was developed by EA Vancouver. Based on internal game database...
------------------------------------------------------------

Q: When was God of War Ragnarok released?


A: God of War Ragnarok was released on November 9, 2022. Based on internal game database...
------------------------------------------------------------

Q: What platform was Pokemon Red launched on?


A: Pokemon Red was launched on the Game Boy. Based on internal game database...
------------------------------------------------------------

Q: What is Rockstar Games working on right now?


A: Rockstar Games is currently working on Grand Theft Auto VI (GTA 6), which was officially revealed with a trailer in December 2023. The game is scheduled to release on November 19, 2026, for PlayStation 5 and Xbox Series X/S. It is set in the fictional state of Leonida, based on Florida, and follows protagonists Jason and Lucia. (Based on internal game database...)
------------------------------------------------------------
